# VectorBT Pro and OSS on Current Case-Study Strategies

This notebook reports the VectorBT Pro and VectorBT OSS rows from the current real-strategy audit.
The current ETF strategy and all required synthetic scenarios match exactly. The remaining
real-strategy failure is the CME equity path under VectorBT Pro.

**Learning objectives**

- Compare VectorBT Pro and OSS with ML4T on the supported ETF strategy
- Distinguish exact fill parity from a small equity-path residual
- Understand why VectorBT OSS is not used for the CME futures contract
- Read engine-only runtime evidence without treating it as a universal ranking

**Book reference**: Chapter 16, Section 16.3

## Setup

In [1]:
"""Current VectorBT parity evidence."""

import json

import polars as pl
from IPython.display import display

from utils.paths import get_chapter_dir

In [2]:
# Production defaults - Papermill injects overrides after this cell
ROUND_SECONDS = 3

In [3]:
AUDIT_PATH = get_chapter_dir(16) / "resources" / "framework_parity_audit.json"
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
FRAMEWORKS = ["vectorbt_pro", "vectorbt_oss"]
FRAMEWORK_NAMES = {
    key: f"{audit['frameworks'][key]['display_name']} {audit['frameworks'][key]['version']}"
    for key in FRAMEWORKS
}
CASE_NAMES = {
    "etfs": "ETF allocation",
    "cme_futures": "CME futures",
    "crypto_perps_funding": "Crypto perpetual funding",
}

## 1. Required comparisons

In [4]:
results = (
    pl.DataFrame(audit["real_strategy_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
    )
    .select(
        "strategy",
        "engine",
        "status",
        "fills",
        "valuations",
        "valuation_timestamps_match",
        "equity_gap",
        "terminal_gap",
    )
    .sort("strategy", "engine")
)

assert results.height == 3
assert results.filter(pl.col("status") == "pass").height == 2
assert results["valuation_timestamps_match"].all()

display(results)

strategy,engine,status,fills,valuations,valuation_timestamps_match,equity_gap,terminal_gap
str,str,str,i64,i64,bool,str,str
"""CME futures""","""VectorBT Pro 2026.6.27""","""fail""",3545,1595,true,"""0.00000010""","""0.00000007"""
"""ETF allocation""","""VectorBT OSS 1.1.0""","""pass""",2466,1995,true,"""0.00000000""","""0.00000000"""
"""ETF allocation""","""VectorBT Pro 2026.6.27""","""pass""",2466,1995,true,"""0.00000000""","""0.00000000"""


Both VectorBT editions reproduce the ETF strategy exactly across 2,466 fills and 1,995 valuations.
VectorBT Pro reproduces all 3,545 CME fills, but the maximum equity gap is `0.00000010` and the
terminal gap is `0.00000007`; the row therefore fails the `1e-8` gate.

## 2. Unsupported asset models

In [5]:
unsupported = (
    pl.DataFrame(audit["unsupported_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
    )
    .select("strategy", "engine", "reason")
    .sort("strategy", "engine")
)
display(unsupported)

strategy,engine,reason
str,str,str
"""CME futures""","""VectorBT OSS 1.1.0""","""no native futures contract mul…"
"""Crypto perpetual funding""","""VectorBT OSS 1.1.0""","""no native perpetual-futures fu…"
"""Crypto perpetual funding""","""VectorBT Pro 2026.6.27""","""no native perpetual-futures fu…"


VectorBT OSS does not provide the native multiplier and margin-account model required by the CME
bundle. Neither edition is used to emulate crypto-perpetual funding and margin accounting.

## 3. Engine-only timing

Only the two correctness-passing ETF rows are timed for publication.

In [6]:
timing = (
    pl.DataFrame(audit["performance_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
        pl.col("framework_median_seconds").round(ROUND_SECONDS).alias("vectorbt_seconds"),
        pl.col("ml4t_median_seconds").round(ROUND_SECONDS).alias("ml4t_seconds"),
        pl.col("framework_to_ml4t_ratio").round(2).alias("vectorbt_div_ml4t"),
    )
    .select("engine", "vectorbt_seconds", "ml4t_seconds", "vectorbt_div_ml4t")
)

assert timing.height == 2
display(timing)

engine,vectorbt_seconds,ml4t_seconds,vectorbt_div_ml4t
str,f64,f64,f64
"""VectorBT Pro 2026.6.27""",0.28,0.454,0.62
"""VectorBT OSS 1.1.0""",0.167,0.453,0.37


Both VectorBT editions have lower median engine-call time on the ETF workload. The measured region
excludes data loading, target construction, adapter preparation, and output extraction. The result
does not establish the same ratio for other datasets or strategy mechanics.

## 4. Synthetic stress evidence

In [7]:
stress = (
    pl.DataFrame(audit["synthetic_stress"]["records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"))
    .select("engine", "intents", "fills", "trades", "terminal_value", "status")
)
display(stress)

engine,intents,fills,trades,terminal_value,status
str,i64,i64,i64,f64,str
"""VectorBT Pro 2026.6.27""",427790,390369,188549,716785.408089,"""pass"""
"""VectorBT OSS 1.1.0""",427790,390369,188549,716785.408089,"""pass"""


VectorBT Pro and OSS both pass the 250-asset, 1.26-million-bar synthetic stress comparison against
their matching ML4T profiles. This establishes scale conformance for the fixed target-order recipe.
It does not overturn the CME real-strategy failure or create support for the excluded asset models.